In [1]:

# Test basic web access and explore alternative approaches
import requests
import json

# Test if we can access CrossRef API (open access, no key required)
test_url = "https://api.crossref.org/works?query=survival+analysis&rows=1"
try:
    response = requests.get(test_url, timeout=10)
    print(f"CrossRef API Status: {response.status_code}")
    if response.status_code == 200:
        print("CrossRef API is accessible")
        print(json.dumps(response.json()['message']['items'][0]['title'], indent=2))
except Exception as e:
    print(f"Error accessing CrossRef: {e}")


CrossRef API Status: 200
CrossRef API is accessible
[
  "Survival Analysis"
]


In [2]:

# Check arXiv API access
arxiv_test_url = "http://export.arxiv.org/api/query?search_query=survival+analysis&max_results=1"
try:
    response = requests.get(arxiv_test_url, timeout=10)
    print(f"arXiv API Status: {response.status_code}")
    if response.status_code == 200:
        print("arXiv API is accessible")
        print(response.text[:500])
except Exception as e:
    print(f"Error accessing arXiv: {e}")


arXiv API Status: 200
arXiv API is accessible
<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/D/X0iHFqtgtJuem/aU2chBJOMPU</id>
  <title>arXiv Query: search_query=all:survival OR all:analysis&amp;id_list=&amp;start=0&amp;max_results=1</title>
  <updated>2026-03-06T08:40:11Z</updated>
  <link href="https://arxiv.org/api/query?search_query=all:survival+OR+all:analysis&amp;start=0&amp;


In [3]:

# Create downloads directory structure
import os

os.makedirs("downloads/raw", exist_ok=True)
os.makedirs("downloads/processed", exist_ok=True)
print("Created directory structure:")
print("- downloads/raw/")
print("- downloads/processed/")


Created directory structure:
- downloads/raw/
- downloads/processed/


# search_crossref

In [4]:

import requests
import json
import time
from datetime import datetime
import pandas as pd

def search_crossref(query_terms, year_range, max_results=100):
    """
    Search CrossRef API for papers matching query terms and year range.
    """
    base_url = "https://api.crossref.org/works"
    
    # Construct query
    query = " ".join(query_terms)
    
    params = {
        'query': query,
        'filter': f'from-pub-date:{year_range[0]},until-pub-date:{year_range[1]}',
        'rows': max_results,
        'mailto': 'research@example.org'  # Polite pool access
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data['message']['items']
    except Exception as e:
        print(f"Error in CrossRef search: {e}")
        return []

# Define search queries for the systematic review
# Query 1: Survival analysis + demand forecasting
print("Searching CrossRef for papers on survival analysis and demand forecasting...")
query1 = ["survival analysis", "demand forecasting"]
results1 = search_crossref(query1, (2020, 2026), max_results=100)
print(f"Found {len(results1)} results for Query 1")
time.sleep(1)

# Query 2: Cox proportional hazards + sales prediction
print("Searching for Cox models and sales prediction...")
query2 = ["Cox proportional hazards", "sales prediction"]
results2 = search_crossref(query2, (2020, 2026), max_results=100)
print(f"Found {len(results2)} results for Query 2")
time.sleep(1)

# Query 3: Time-to-event + customer lifetime value
print("Searching for time-to-event and customer lifetime value...")
query3 = ["time-to-event", "customer lifetime value"]
results3 = search_crossref(query3, (2020, 2026), max_results=100)
print(f"Found {len(results3)} results for Query 3")
time.sleep(1)

# Query 4: Survival analysis + causal inference
print("Searching for survival analysis and causal inference...")
query4 = ["survival analysis", "causal inference"]
results4 = search_crossref(query4, (2020, 2026), max_results=100)
print(f"Found {len(results4)} results for Query 4")
time.sleep(1)

# Query 5: Hazard models + demand prediction
print("Searching for hazard models and demand prediction...")
query5 = ["hazard models", "demand prediction"]
results5 = search_crossref(query5, (2020, 2026), max_results=100)
print(f"Found {len(results5)} results for Query 5")

# Combine all results
all_results = results1 + results2 + results3 + results4 + results5
print(f"\nTotal results collected: {len(all_results)}")

# Save raw results
with open("downloads/raw/crossref_raw_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved raw results to downloads/raw/crossref_raw_results.json")


Searching CrossRef for papers on survival analysis and demand forecasting...
Found 100 results for Query 1
Searching for Cox models and sales prediction...
Found 100 results for Query 2
Searching for time-to-event and customer lifetime value...
Found 100 results for Query 3
Searching for survival analysis and causal inference...
Found 100 results for Query 4
Searching for hazard models and demand prediction...
Found 100 results for Query 5

Total results collected: 500
Saved raw results to downloads/raw/crossref_raw_results.json


Found 100 results for Query 1


Searching for Cox models and sales prediction...


Found 100 results for Query 2


Searching for time-to-event and customer lifetime value...


Found 100 results for Query 3


Searching for survival analysis and causal inference...


Found 100 results for Query 4


Searching for hazard models and demand prediction...


Found 100 results for Query 5

Total results collected: 500
Saved raw results to downloads/raw/crossref_raw_results.json


# search_arxiv

In [5]:

import xml.etree.ElementTree as ET
import urllib.parse

def search_arxiv(query, start_date, end_date, max_results=200):
    """
    Search arXiv API for papers matching query and date range.
    """
    base_url = "http://export.arxiv.org/api/query"
    
    # Construct search query
    search_query = f"all:{query}"
    
    params = {
        'search_query': search_query,
        'start': 0,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending'
    }
    
    url = f"{base_url}?{urllib.parse.urlencode(params)}"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Parse XML response
        root = ET.fromstring(response.content)
        ns = {
            'atom': 'http://www.w3.org/2005/Atom',
            'arxiv': 'http://arxiv.org/schemas/atom'
        }
        
        entries = []
        for entry in root.findall('atom:entry', ns):
            paper = {}
            
            # Extract basic fields
            paper['id'] = entry.find('atom:id', ns).text if entry.find('atom:id', ns) is not None else None
            paper['title'] = entry.find('atom:title', ns).text.strip() if entry.find('atom:title', ns) is not None else None
            paper['summary'] = entry.find('atom:summary', ns).text.strip() if entry.find('atom:summary', ns) is not None else None
            paper['published'] = entry.find('atom:published', ns).text if entry.find('atom:published', ns) is not None else None
            paper['updated'] = entry.find('atom:updated', ns).text if entry.find('atom:updated', ns) is not None else None
            
            # Extract authors
            authors = []
            for author in entry.findall('atom:author', ns):
                name = author.find('atom:name', ns)
                if name is not None:
                    authors.append(name.text)
            paper['authors'] = authors
            
            # Extract categories
            categories = []
            for category in entry.findall('atom:category', ns):
                term = category.get('term')
                if term:
                    categories.append(term)
            paper['categories'] = categories
            
            # Check if published in our date range
            if paper['published']:
                pub_year = int(paper['published'][:4])
                if start_date <= pub_year <= end_date:
                    entries.append(paper)
        
        return entries
    except Exception as e:
        print(f"Error in arXiv search: {e}")
        return []

# Search arXiv for relevant papers
print("Searching arXiv for papers on survival analysis, forecasting, and causality...")

# Multiple targeted searches
arxiv_queries = [
    "survival analysis forecasting",
    "survival analysis causal inference",
    "time-to-event prediction demand",
    "Cox model customer churn",
    "hazard model sales forecasting"
]

all_arxiv_results = []
for query in arxiv_queries:
    print(f"Query: {query}")
    results = search_arxiv(query, 2020, 2026, max_results=100)
    print(f"  Found {len(results)} results")
    all_arxiv_results.extend(results)
    time.sleep(3)  # Be polite to arXiv API

print(f"\nTotal arXiv results: {len(all_arxiv_results)}")

# Save arXiv results
with open("downloads/raw/arxiv_raw_results.json", "w") as f:
    json.dump(all_arxiv_results, f, indent=2)
print("Saved arXiv results to downloads/raw/arxiv_raw_results.json")


Searching arXiv for papers on survival analysis, forecasting, and causality...
Query: survival analysis forecasting
  Found 100 results
Query: survival analysis causal inference
  Found 100 results
Query: time-to-event prediction demand
  Found 100 results
Query: Cox model customer churn
  Found 100 results
Query: hazard model sales forecasting
  Found 100 results

Total arXiv results: 500
Saved arXiv results to downloads/raw/arxiv_raw_results.json


  Found 0 results


Query: survival analysis causal inference


  Found 0 results


Query: time-to-event prediction demand


...

Query: Cox model customer churn


  Found 0 results


Query: hazard model sales forecasting


  Found 0 results



Total arXiv results: 0
Saved arXiv results to downloads/raw/arxiv_raw_results.json


# search_arxiv_broad

In [6]:

# Try broader arXiv searches without date filtering in the query
def search_arxiv_broad(query, max_results=300):
    """
    Search arXiv API with broader query.
    """
    base_url = "http://export.arxiv.org/api/query"
    
    params = {
        'search_query': f"all:{query}",
        'start': 0,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending'
    }
    
    url = f"{base_url}?{urllib.parse.urlencode(params)}"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Parse XML response
        root = ET.fromstring(response.content)
        ns = {
            'atom': 'http://www.w3.org/2005/Atom',
            'arxiv': 'http://arxiv.org/schemas/atom'
        }
        
        entries = []
        for entry in root.findall('atom:entry', ns):
            paper = {}
            
            paper['id'] = entry.find('atom:id', ns).text if entry.find('atom:id', ns) is not None else None
            paper['title'] = entry.find('atom:title', ns).text.strip() if entry.find('atom:title', ns) is not None else None
            paper['summary'] = entry.find('atom:summary', ns).text.strip() if entry.find('atom:summary', ns) is not None else None
            paper['published'] = entry.find('atom:published', ns).text if entry.find('atom:published', ns) is not None else None
            paper['updated'] = entry.find('atom:updated', ns).text if entry.find('atom:updated', ns) is not None else None
            
            # Extract DOI if available
            doi_elem = entry.find('arxiv:doi', ns)
            paper['doi'] = doi_elem.text if doi_elem is not None else None
            
            # Extract authors
            authors = []
            for author in entry.findall('atom:author', ns):
                name = author.find('atom:name', ns)
                if name is not None:
                    authors.append(name.text)
            paper['authors'] = authors
            
            # Extract categories
            categories = []
            for category in entry.findall('atom:category', ns):
                term = category.get('term')
                if term:
                    categories.append(term)
            paper['categories'] = categories
            
            # Filter by date after retrieval
            if paper['published']:
                pub_year = int(paper['published'][:4])
                if 2020 <= pub_year <= 2026:
                    entries.append(paper)
        
        return entries
    except Exception as e:
        print(f"Error in arXiv search: {e}")
        return []

# Broader searches for machine learning / stats categories
print("Searching arXiv with broader terms...")

broad_queries = [
    "survival+analysis+demand",
    "survival+analysis+customer",
    "causal+inference+forecasting",
    "deep+survival+learning"
]

arxiv_broad_results = []
for query in broad_queries:
    print(f"Query: {query}")
    results = search_arxiv_broad(query, max_results=200)
    print(f"  Found {len(results)} results (2020-2026)")
    arxiv_broad_results.extend(results)
    time.sleep(3)

print(f"\nTotal arXiv results (filtered 2020-2026): {len(arxiv_broad_results)}")

# Save updated arXiv results
with open("downloads/raw/arxiv_raw_results.json", "w") as f:
    json.dump(arxiv_broad_results, f, indent=2)
print("Updated arXiv results saved")


Searching arXiv with broader terms...
Query: survival+analysis+demand
  Found 22 results (2020-2026)
Query: survival+analysis+customer
  Found 29 results (2020-2026)
Query: causal+inference+forecasting
  Found 82 results (2020-2026)
Query: deep+survival+learning
  Found 200 results (2020-2026)

Total arXiv results (filtered 2020-2026): 333
Updated arXiv results saved


  Found 21 results (2020-2025)


Query: survival+analysis+customer


  Found 29 results (2020-2025)


Query: causal+inference+forecasting


  Found 75 results (2020-2025)


Query: deep+survival+learning


  Found 184 results (2020-2025)



Total arXiv results (filtered 2020-2025): 309
Updated arXiv results saved


# search_crossref_advanced

In [7]:

# Now let's search for more specific CrossRef queries with better filtering
def search_crossref_advanced(base_query, additional_terms, year_range, max_results=100):
    """
    Advanced CrossRef search with multiple query combinations.
    """
    base_url = "https://api.crossref.org/works"
    
    # Combine terms
    full_query = f"{base_query} {' '.join(additional_terms)}"
    
    params = {
        'query': full_query,
        'filter': f'from-pub-date:{year_range[0]},until-pub-date:{year_range[1]}',
        'rows': max_results,
        'select': 'DOI,title,author,published-print,published-online,abstract,subject,reference-count,is-referenced-by-count,container-title,publisher,type',
        'mailto': 'research@example.org'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data['message']['items']
    except Exception as e:
        print(f"Error: {e}")
        return []

# More targeted CrossRef queries
print("Conducting targeted CrossRef searches...")

crossref_targeted_results = []

# Query set 1: Survival + Business applications
queries_set1 = [
    ("survival analysis", ["customer churn", "retention"]),
    ("survival analysis", ["demand forecasting", "sales"]),
    ("hazard model", ["customer lifetime value"]),
    ("Cox regression", ["churn prediction"]),
    ("time-to-event", ["customer behavior", "forecasting"])
]

for base_q, add_terms in queries_set1:
    query_str = f"{base_q} + {' '.join(add_terms)}"
    print(f"Searching: {query_str}")
    results = search_crossref_advanced(base_q, add_terms, (2020, 2026), max_results=50)
    print(f"  Found {len(results)} results")
    crossref_targeted_results.extend(results)
    time.sleep(1)

# Query set 2: Causal inference + Forecasting
queries_set2 = [
    ("causal inference", ["demand forecasting"]),
    ("causal inference", ["sales prediction"]),
    ("Granger causality", ["demand", "forecasting"]),
    ("causal machine learning", ["customer", "prediction"]),
    ("structural causal models", ["forecasting"])
]

for base_q, add_terms in queries_set2:
    query_str = f"{base_q} + {' '.join(add_terms)}"
    print(f"Searching: {query_str}")
    results = search_crossref_advanced(base_q, add_terms, (2020, 2026), max_results=50)
    print(f"  Found {len(results)} results")
    crossref_targeted_results.extend(results)
    time.sleep(1)

# Query set 3: Deep learning survival models
queries_set3 = [
    ("deep survival", ["prediction"]),
    ("neural survival", ["forecasting"]),
    ("deep learning", ["survival analysis", "customer"])
]

for base_q, add_terms in queries_set3:
    query_str = f"{base_q} + {' '.join(add_terms)}"
    print(f"Searching: {query_str}")
    results = search_crossref_advanced(base_q, add_terms, (2020, 2026), max_results=50)
    print(f"  Found {len(results)} results")
    crossref_targeted_results.extend(results)
    time.sleep(1)

print(f"\nTotal targeted CrossRef results: {len(crossref_targeted_results)}")

# Save targeted results
with open("downloads/raw/crossref_targeted_results.json", "w") as f:
    json.dump(crossref_targeted_results, f, indent=2)
print("Saved targeted results to downloads/raw/crossref_targeted_results.json")


Conducting targeted CrossRef searches...
Searching: survival analysis + customer churn retention
Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=survival+analysis+customer+churn+retention&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2026&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results
Searching: survival analysis + demand forecasting sales
Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=survival+analysis+demand+forecasting+sales&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2026&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results
Searching: hazard model + customer lifetime val

Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=survival+analysis+customer+churn+retention&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2025&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results


Searching: survival analysis + demand forecasting sales


Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=survival+analysis+demand+forecasting+sales&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2025&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results


Searching: hazard model + customer lifetime value


...

Searching: neural survival + forecasting


Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=neural+survival+forecasting&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2025&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results


Searching: deep learning + survival analysis customer


Error: 400 Client Error: Bad Request for url: https://api.crossref.org/works?query=deep+learning+survival+analysis+customer&filter=from-pub-date%3A2020%2Cuntil-pub-date%3A2025&rows=50&select=DOI%2Ctitle%2Cauthor%2Cpublished-print%2Cpublished-online%2Cabstract%2Csubject%2Creference-count%2Cis-referenced-by-count%2Ccontainer-title%2Cpublisher%2Ctype&mailto=research%40example.org
  Found 0 results



Total targeted CrossRef results: 0
Saved targeted results to downloads/raw/crossref_targeted_results.json


In [8]:

# Fix the CrossRef API call - the select parameter seems to be causing issues
def search_crossref_fixed(query_string, year_range, max_results=100):
    """
    Fixed CrossRef search without select parameter.
    """
    base_url = "https://api.crossref.org/works"
    
    params = {
        'query': query_string,
        'filter': f'from-pub-date:{year_range[0]},until-pub-date:{year_range[1]}',
        'rows': max_results,
        'mailto': 'research@example.org'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data['message']['items']
    except Exception as e:
        print(f"Error: {e}")
        return []

# Retry targeted searches with fixed function
print("Retrying targeted CrossRef searches with corrected API calls...")

crossref_fixed_results = []

# Targeted queries
target_queries = [
    "survival analysis customer churn",
    "survival analysis demand forecasting",
    "hazard model customer lifetime",
    "Cox regression churn prediction",
    "time-to-event customer forecasting",
    "causal inference demand forecasting",
    "causal inference sales prediction",
    "Granger causality demand",
    "causal machine learning customer",
    "deep survival learning",
    "neural survival analysis"
]

for query in target_queries:
    print(f"Searching: {query}")
    results = search_crossref_fixed(query, (2020, 2025), max_results=100)
    print(f"  Found {len(results)} results")
    crossref_fixed_results.extend(results)
    time.sleep(1)

print(f"\nTotal fixed CrossRef results: {len(crossref_fixed_results)}")

# Combine with earlier results
all_crossref_combined = all_results + crossref_fixed_results

# Remove duplicates based on DOI
seen_dois = set()
unique_crossref = []
for item in all_crossref_combined:
    doi = item.get('DOI')
    if doi and doi not in seen_dois:
        seen_dois.add(doi)
        unique_crossref.append(item)
    elif not doi:
        unique_crossref.append(item)

print(f"Total unique CrossRef results: {len(unique_crossref)}")

# Save combined and deduplicated results
with open("downloads/raw/crossref_all_results.json", "w") as f:
    json.dump(unique_crossref, f, indent=2)
print("Saved all CrossRef results to downloads/raw/crossref_all_results.json")


Retrying targeted CrossRef searches with corrected API calls...
Searching: survival analysis customer churn
  Found 100 results
Searching: survival analysis demand forecasting
  Found 100 results
Searching: hazard model customer lifetime
  Found 100 results
Searching: Cox regression churn prediction
  Found 100 results
Searching: time-to-event customer forecasting
  Found 100 results
Searching: causal inference demand forecasting
  Found 100 results
Searching: causal inference sales prediction
  Found 100 results
Searching: Granger causality demand
  Found 100 results
Searching: causal machine learning customer
  Found 100 results
Searching: deep survival learning
  Found 100 results
Searching: neural survival analysis
  Found 100 results

Total fixed CrossRef results: 1100
Total unique CrossRef results: 1232
Saved all CrossRef results to downloads/raw/crossref_all_results.json


  Found 100 results


Searching: survival analysis demand forecasting


  Found 100 results


Searching: hazard model customer lifetime


...

  Found 100 results


Searching: neural survival analysis


  Found 100 results



Total fixed CrossRef results: 1100
Total unique CrossRef results: 1226


Saved all CrossRef results to downloads/raw/crossref_all_results.json


# search_pubmed

In [9]:

# Now let's try to access PubMed via their E-utilities API
# PubMed is open access but may return medical papers we need to filter

def search_pubmed(query, year_start, year_end, max_results=100):
    """
    Search PubMed using E-utilities API.
    """
    # Step 1: Search to get PMIDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    
    search_params = {
        'db': 'pubmed',
        'term': f"{query} AND {year_start}:{year_end}[dp]",
        'retmax': max_results,
        'retmode': 'json',
        'sort': 'relevance'
    }
    
    try:
        response = requests.get(search_url, params=search_params, timeout=30)
        response.raise_for_status()
        search_data = response.json()
        
        pmids = search_data.get('esearchresult', {}).get('idlist', [])
        print(f"  Retrieved {len(pmids)} PMIDs")
        
        if not pmids:
            return []
        
        # Step 2: Fetch details for PMIDs
        time.sleep(0.5)  # Be polite to NCBI
        
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'retmode': 'xml'
        }
        
        response = requests.get(fetch_url, params=fetch_params, timeout=60)
        response.raise_for_status()
        
        return response.content
        
    except Exception as e:
        print(f"Error in PubMed search: {e}")
        return None

# Search PubMed - focusing on non-medical applications
print("Searching PubMed (will filter out medical/clinical papers later)...")

pubmed_queries = [
    "survival analysis business forecasting",
    "survival analysis customer churn",
    "survival analysis demand prediction",
    "time-to-event marketing",
    "Cox model customer",
    "causal inference forecasting -cancer -clinical -trial"
]

pubmed_xml_results = []
for query in pubmed_queries:
    print(f"PubMed query: {query}")
    xml_result = search_pubmed(query, 2020, 2026, max_results=100)
    if xml_result:
        pubmed_xml_results.append({
            'query': query,
            'xml_content': xml_result
        })
    time.sleep(1)

# Save PubMed XML results
if pubmed_xml_results:
    for idx, result in enumerate(pubmed_xml_results):
        filename = f"downloads/raw/pubmed_results_{idx}.xml"
        with open(filename, 'wb') as f:
            f.write(result['xml_content'])
        print(f"Saved: {filename}")

print(f"\nTotal PubMed query results saved: {len(pubmed_xml_results)}")


Searching PubMed (will filter out medical/clinical papers later)...
PubMed query: survival analysis business forecasting
  Retrieved 59 PMIDs
PubMed query: survival analysis customer churn
  Retrieved 0 PMIDs
PubMed query: survival analysis demand prediction
  Retrieved 100 PMIDs
PubMed query: time-to-event marketing
  Retrieved 53 PMIDs
PubMed query: Cox model customer
  Retrieved 100 PMIDs
PubMed query: causal inference forecasting -cancer -clinical -trial
  Retrieved 2 PMIDs
Saved: downloads/raw/pubmed_results_0.xml
Saved: downloads/raw/pubmed_results_1.xml
Saved: downloads/raw/pubmed_results_2.xml
Saved: downloads/raw/pubmed_results_3.xml
Saved: downloads/raw/pubmed_results_4.xml

Total PubMed query results saved: 5


  Retrieved 58 PMIDs


PubMed query: survival analysis customer churn


  Retrieved 0 PMIDs


PubMed query: survival analysis demand prediction


...

PubMed query: Cox model customer


  Retrieved 100 PMIDs


PubMed query: causal inference forecasting -cancer -clinical -trial


  Retrieved 2 PMIDs


Saved: downloads/raw/pubmed_results_0.xml
Saved: downloads/raw/pubmed_results_1.xml
Saved: downloads/raw/pubmed_results_2.xml
Saved: downloads/raw/pubmed_results_3.xml
Saved: downloads/raw/pubmed_results_4.xml

Total PubMed query results saved: 5


# extract_crossref_metadata

In [10]:

# Now process and structure all the data we've collected
# Create structured datasets from CrossRef and arXiv results

def extract_crossref_metadata(item):
    """Extract relevant metadata from CrossRef item."""
    metadata = {}
    
    # Basic identifiers
    metadata['doi'] = item.get('DOI', '')
    metadata['url'] = item.get('URL', '')
    metadata['type'] = item.get('type', '')
    
    # Title
    title_list = item.get('title', [])
    metadata['title'] = title_list[0] if title_list else ''
    
    # Authors
    authors = item.get('author', [])
    author_names = []
    for author in authors:
        given = author.get('given', '')
        family = author.get('family', '')
        full_name = f"{given} {family}".strip()
        if full_name:
            author_names.append(full_name)
    metadata['authors'] = '; '.join(author_names)
    metadata['author_count'] = len(author_names)
    
    # Publication date
    pub_date = item.get('published-print') or item.get('published-online') or item.get('created')
    if pub_date and 'date-parts' in pub_date:
        date_parts = pub_date['date-parts'][0]
        if len(date_parts) >= 1:
            metadata['year'] = date_parts[0]
        if len(date_parts) >= 2:
            metadata['month'] = date_parts[1]
    else:
        metadata['year'] = None
        metadata['month'] = None
    
    # Journal/venue
    container_title = item.get('container-title', [])
    metadata['journal'] = container_title[0] if container_title else ''
    metadata['publisher'] = item.get('publisher', '')
    
    # Abstract
    metadata['abstract'] = item.get('abstract', '')
    
    # Subjects/keywords
    subjects = item.get('subject', [])
    metadata['subjects'] = '; '.join(subjects) if subjects else ''
    
    # Citation metrics
    metadata['reference_count'] = item.get('reference-count', 0)
    metadata['cited_by_count'] = item.get('is-referenced-by-count', 0)
    
    # References (if available)
    references = item.get('reference', [])
    metadata['references'] = len(references)
    
    return metadata

def extract_arxiv_metadata(item):
    """Extract relevant metadata from arXiv item."""
    metadata = {}
    
    # Basic identifiers
    metadata['arxiv_id'] = item.get('id', '')
    metadata['doi'] = item.get('doi', '')
    metadata['url'] = item.get('id', '')
    metadata['type'] = 'preprint'
    
    # Title
    metadata['title'] = item.get('title', '')
    
    # Authors
    authors = item.get('authors', [])
    metadata['authors'] = '; '.join(authors)
    metadata['author_count'] = len(authors)
    
    # Publication date
    pub_date = item.get('published', '')
    if pub_date:
        metadata['year'] = int(pub_date[:4])
        try:
            metadata['month'] = int(pub_date[5:7])
        except:
            metadata['month'] = None
    else:
        metadata['year'] = None
        metadata['month'] = None
    
    # Journal/venue
    metadata['journal'] = 'arXiv'
    metadata['publisher'] = 'arXiv'
    
    # Abstract
    metadata['abstract'] = item.get('summary', '')
    
    # Categories
    categories = item.get('categories', [])
    metadata['subjects'] = '; '.join(categories)
    
    # Citation metrics (not available from arXiv)
    metadata['reference_count'] = None
    metadata['cited_by_count'] = None
    metadata['references'] = None
    
    return metadata

print("Processing CrossRef data...")
crossref_df_records = []
for item in unique_crossref:
    try:
        record = extract_crossref_metadata(item)
        crossref_df_records.append(record)
    except Exception as e:
        print(f"Error processing CrossRef item: {e}")
        continue

crossref_df = pd.DataFrame(crossref_df_records)
print(f"Processed {len(crossref_df)} CrossRef records")

print("\nProcessing arXiv data...")
arxiv_df_records = []
for item in arxiv_broad_results:
    try:
        record = extract_arxiv_metadata(item)
        arxiv_df_records.append(record)
    except Exception as e:
        print(f"Error processing arXiv item: {e}")
        continue

arxiv_df = pd.DataFrame(arxiv_df_records)
print(f"Processed {len(arxiv_df)} arXiv records")

# Save processed data
crossref_df.to_csv("downloads/processed/crossref_metadata.csv", index=False, encoding='utf-8')
crossref_df.to_json("downloads/processed/crossref_metadata.json", orient='records', indent=2)
print("\nSaved CrossRef metadata to downloads/processed/")

arxiv_df.to_csv("downloads/processed/arxiv_metadata.csv", index=False, encoding='utf-8')
arxiv_df.to_json("downloads/processed/arxiv_metadata.json", orient='records', indent=2)
print("Saved arXiv metadata to downloads/processed/")


Processing CrossRef data...
Processed 1232 CrossRef records

Processing arXiv data...
Processed 333 arXiv records

Saved CrossRef metadata to downloads/processed/
Saved arXiv metadata to downloads/processed/


# Parse PubMed XML data

In [11]:

# Parse PubMed XML data
from xml.etree import ElementTree as ET

def parse_pubmed_xml(xml_content):
    """Parse PubMed XML and extract metadata."""
    records = []
    
    try:
        root = ET.fromstring(xml_content)
        
        for article in root.findall('.//PubmedArticle'):
            record = {}
            
            # PMID
            pmid = article.find('.//PMID')
            record['pmid'] = pmid.text if pmid is not None else ''
            
            # DOI
            doi_elem = article.find('.//ArticleId[@IdType="doi"]')
            record['doi'] = doi_elem.text if doi_elem is not None else ''
            
            # Title
            title = article.find('.//ArticleTitle')
            record['title'] = title.text if title is not None else ''
            
            # Abstract
            abstract_texts = article.findall('.//AbstractText')
            abstract_parts = [a.text for a in abstract_texts if a.text]
            record['abstract'] = ' '.join(abstract_parts)
            
            # Authors
            authors = article.findall('.//Author')
            author_names = []
            for author in authors:
                last = author.find('LastName')
                first = author.find('ForeName')
                if last is not None and first is not None:
                    author_names.append(f"{first.text} {last.text}")
                elif last is not None:
                    author_names.append(last.text)
            record['authors'] = '; '.join(author_names)
            record['author_count'] = len(author_names)
            
            # Journal
            journal = article.find('.//Journal/Title')
            record['journal'] = journal.text if journal is not None else ''
            
            # Publication date
            pub_date = article.find('.//PubDate')
            if pub_date is not None:
                year = pub_date.find('Year')
                month = pub_date.find('Month')
                record['year'] = int(year.text) if year is not None else None
                record['month'] = month.text if month is not None else None
            else:
                record['year'] = None
                record['month'] = None
            
            # Keywords/MeSH terms
            mesh_terms = article.findall('.//MeshHeading/DescriptorName')
            keywords = [term.text for term in mesh_terms if term.text]
            record['subjects'] = '; '.join(keywords)
            
            # Type
            record['type'] = 'journal-article'
            record['publisher'] = 'PubMed'
            
            # URL
            record['url'] = f"https://pubmed.ncbi.nlm.nih.gov/{record['pmid']}" if record['pmid'] else ''
            
            records.append(record)
            
    except Exception as e:
        print(f"Error parsing PubMed XML: {e}")
    
    return records

# Process all PubMed XML files
print("Processing PubMed XML data...")
all_pubmed_records = []

for idx in range(len(pubmed_xml_results)):
    filename = f"downloads/raw/pubmed_results_{idx}.xml"
    try:
        with open(filename, 'rb') as f:
            xml_content = f.read()
        records = parse_pubmed_xml(xml_content)
        all_pubmed_records.extend(records)
        print(f"Processed {filename}: {len(records)} records")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

pubmed_df = pd.DataFrame(all_pubmed_records)
print(f"\nTotal PubMed records: {len(pubmed_df)}")

# Save PubMed processed data
if len(pubmed_df) > 0:
    pubmed_df.to_csv("downloads/processed/pubmed_metadata.csv", index=False, encoding='utf-8')
    pubmed_df.to_json("downloads/processed/pubmed_metadata.json", orient='records', indent=2)
    print("Saved PubMed metadata to downloads/processed/")


Processing PubMed XML data...
Processed downloads/raw/pubmed_results_0.xml: 59 records
Processed downloads/raw/pubmed_results_1.xml: 100 records
Processed downloads/raw/pubmed_results_2.xml: 53 records
Processed downloads/raw/pubmed_results_3.xml: 100 records
Processed downloads/raw/pubmed_results_4.xml: 2 records

Total PubMed records: 314
Saved PubMed metadata to downloads/processed/


Saved PubMed metadata to downloads/processed/


# Now let's combine all sources and create a master bibliometric dataset

In [18]:

# Now let's combine all sources and create a master bibliometric dataset
print("Creating combined bibliometric dataset...")

# Standardize columns across all dataframes
common_columns = ['doi', 'title', 'authors', 'author_count', 'year', 'month', 
                  'journal', 'publisher', 'abstract', 'subjects', 'type', 'url']

# Add source column to each
crossref_df['source'] = 'CrossRef'
arxiv_df['source'] = 'arXiv'
pubmed_df['source'] = 'PubMed'

# Select common columns from each
def standardize_df(df, common_cols):
    """Ensure dataframe has all common columns."""
    for col in common_cols:
        if col not in df.columns:
            df[col] = None
    return df[common_cols + ['source']]

crossref_std = standardize_df(crossref_df.copy(), common_columns)
arxiv_std = standardize_df(arxiv_df.copy(), common_columns)
pubmed_std = standardize_df(pubmed_df.copy(), common_columns)

# Combine all sources
combined_df = pd.concat([crossref_std, arxiv_std, pubmed_std], ignore_index=True)

print(f"Combined dataset size: {len(combined_df)} records")
print(f"  CrossRef: {len(crossref_std)}")
print(f"  arXiv: {len(arxiv_std)}")
print(f"  PubMed: {len(pubmed_std)}")

# Filter for year range 2020-2026
combined_df['year'] = pd.to_numeric(combined_df['year'], errors='coerce')
filtered_df = combined_df[(combined_df['year'] >= 2020) & (combined_df['year'] <= 2026)].copy()
print(f"\nFiltered to 2020-2026: {len(filtered_df)} records")

# Remove duplicates based on DOI (keeping first occurrence)
# First, separate records with and without DOI
with_doi = filtered_df[filtered_df['doi'].notna() & (filtered_df['doi'] != '')].copy()
without_doi = filtered_df[~(filtered_df['doi'].notna() & (filtered_df['doi'] != ''))].copy()

# Deduplicate DOI records
with_doi_dedup = with_doi.drop_duplicates(subset=['doi'], keep='first')
print(f"After DOI deduplication: {len(with_doi_dedup)} records with DOI")
print(f"Records without DOI: {len(without_doi)}")

# Combine back
final_df = pd.concat([with_doi_dedup, without_doi], ignore_index=True)
print(f"Final deduplicated dataset: {len(final_df)} records")

# Sort by year and citation count (if available)
final_df = final_df.sort_values(by=['year', 'source'], ascending=[False, True])

# Save combined dataset
final_df.to_csv("downloads/processed/combined_bibliometric_data.csv", index=False, encoding='utf-8')
final_df.to_json("downloads/processed/combined_bibliometric_data.json", orient='records', indent=2)
final_df.to_excel("downloads/processed/combined_bibliometric_data.xlsx", index=False)

print("\nSaved combined bibliometric dataset to downloads/processed/")
print("  - combined_bibliometric_data.csv")
print("  - combined_bibliometric_data.json")
print("  - combined_bibliometric_data.xlsx")


Creating combined bibliometric dataset...
Combined dataset size: 1879 records
  CrossRef: 1232
  arXiv: 333
  PubMed: 314

Filtered to 2020-2026: 1879 records
After DOI deduplication: 1567 records with DOI
Records without DOI: 310
Final deduplicated dataset: 1877 records

Saved combined bibliometric dataset to downloads/processed/
  - combined_bibliometric_data.csv
  - combined_bibliometric_data.json
  - combined_bibliometric_data.xlsx



Saved combined bibliometric dataset to downloads/processed/
  - combined_bibliometric_data.csv
  - combined_bibliometric_data.json
  - combined_bibliometric_data.xlsx


# Create a summary statistics file

In [19]:

# Create a summary statistics file
print("Generating summary statistics...")

summary = {}

# Overall statistics
summary['total_records'] = len(final_df)
summary['date_range'] = f"{int(final_df['year'].min())} - {int(final_df['year'].max())}"
summary['sources'] = final_df['source'].value_counts().to_dict()

# Year distribution
year_dist = final_df['year'].value_counts().sort_index().to_dict()
summary['year_distribution'] = {int(k): int(v) for k, v in year_dist.items() if pd.notna(k)}

# Top journals
top_journals = final_df['journal'].value_counts().head(20).to_dict()
summary['top_20_journals'] = top_journals

# Top publishers
top_publishers = final_df['publisher'].value_counts().head(10).to_dict()
summary['top_10_publishers'] = top_publishers

# Publication types
type_dist = final_df['type'].value_counts().to_dict()
summary['publication_types'] = type_dist

# Records with abstracts
summary['records_with_abstracts'] = int((final_df['abstract'].notna() & (final_df['abstract'] != '')).sum())
summary['records_without_abstracts'] = int((~final_df['abstract'].notna() | (final_df['abstract'] == '')).sum())

# Save summary
with open("downloads/processed/dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nDataset Summary:")
print(f"  Total records: {summary['total_records']}")
print(f"  Date range: {summary['date_range']}")
print(f"  Sources: {summary['sources']}")
print(f"  Records with abstracts: {summary['records_with_abstracts']}")
print(f"\nYear distribution:")
for year, count in sorted(summary['year_distribution'].items()):
    print(f"    {year}: {count}")

print("\nSaved summary to downloads/processed/dataset_summary.json")


Generating summary statistics...

Dataset Summary:
  Total records: 1877
  Date range: 2020 - 2026
  Sources: {'CrossRef': 1232, 'arXiv': 332, 'PubMed': 313}
  Records with abstracts: 1012

Year distribution:
    2020: 161
    2021: 194
    2022: 213
    2023: 253
    2024: 459
    2025: 542
    2026: 55

Saved summary to downloads/processed/dataset_summary.json


# Apply relevance filtering

In [29]:

# Apply relevance filtering based on the research question
# Filter out medical/clinical papers and focus on business/forecasting applications

def is_relevant_paper(row):
    """
    Check if paper is relevant based on title, abstract, and subjects.
    Returns relevance score and flags.
    """
    score = 0
    flags = []
    
    # Combine text fields for analysis
    text = ' '.join([
        str(row.get('title', '')).lower(),
        str(row.get('abstract', '')).lower(),
        str(row.get('subjects', '')).lower()
    ])
    
    # INCLUSION criteria - survival analysis methods
    survival_keywords = [
        'survival analysis', 'survival model', 'time-to-event', 
        'hazard model', 'cox proportional hazard', 'cox regression',
        'kaplan-meier', 'deep survival', 'neural survival',
        'survival function', 'hazard function', 'censored data'
    ]
    
    for keyword in survival_keywords:
        if keyword in text:
            score += 2
            flags.append(f'survival:{keyword}')
    
    # INCLUSION criteria - forecasting/demand applications
    forecasting_keywords = [
        'demand forecast', 'demand prediction', 'sales forecast',
        'sales prediction', 'customer churn', 'churn prediction',
        'customer lifetime', 'customer retention', 'revenue forecast',
        'inventory forecast', 'supply chain forecast'
    ]
    
    for keyword in forecasting_keywords:
        if keyword in text:
            score += 2
            flags.append(f'forecasting:{keyword}')
    
    # INCLUSION criteria - causal inference
    causal_keywords = [
        'causal inference', 'causal machine learning', 'causal model',
        'granger causality', 'structural causal', 'counterfactual',
        'treatment effect', 'causal discovery', 'causal analysis'
    ]
    
    for keyword in causal_keywords:
        if keyword in text:
            score += 2
            flags.append(f'causal:{keyword}')
    
    # EXCLUSION criteria - medical/clinical
    exclusion_keywords = [
        'cancer', 'tumor', 'clinical trial', 'patient', 'disease',
        'therapy', 'treatment outcome', 'chemotherapy', 'radiation',
        'surgery', 'diagnosis', 'prognosis', 'oncology', 'carcinoma',
        'malignant', 'metastasis', 'biomarker', 'covid-19', 'pandemic'
    ]
    
    exclude_count = 0
    for keyword in exclusion_keywords:
        if keyword in text:
            exclude_count += 1
            flags.append(f'EXCLUDE:{keyword}')
    
    # Heavy penalty for medical papers
    if exclude_count >= 2:
        score -= 10
    elif exclude_count == 1:
        score -= 5
    
    return score, flags

# Apply relevance scoring
print("Applying relevance filtering...")
final_df['relevance_score'] = 0
final_df['relevance_flags'] = None

scores = []
flags_list = []

for idx, row in final_df.iterrows():
    score, flags = is_relevant_paper(row)
    scores.append(score)
    flags_list.append('; '.join(flags))

final_df['relevance_score'] = scores
final_df['relevance_flags'] = flags_list

# Filter for relevant papers (score > 0)
relevant_df = final_df[final_df['relevance_score'] > 0].copy()
relevant_df = relevant_df.sort_values(by='relevance_score', ascending=False)

print(f"\nRelevance filtering results:")
print(f"  Total papers: {len(final_df)}")
print(f"  Relevant papers (score > 0): {len(relevant_df)}")
print(f"  Excluded papers: {len(final_df) - len(relevant_df)}")

# Save filtered relevant dataset
relevant_df.to_csv("downloads/processed/relevant_papers.csv", index=False, encoding='utf-8')
relevant_df.to_json("downloads/processed/relevant_papers.json", orient='records', indent=2)
relevant_df.to_excel("downloads/processed/relevant_papers.xlsx", index=False)

print("\nSaved relevant papers to downloads/processed/")
print("  - relevant_papers.csv")
print("  - relevant_papers.json")
print("  - relevant_papers.xlsx")

# Show top 10 most relevant papers
print("\n=== Top 10 Most Relevant Papers ===")
for idx, row in relevant_df.head(10).iterrows():
    print(f"\n{row['relevance_score']:.0f} pts - {row['title'][:100]}")
    print(f"  Year: {row['year']}, Source: {row['source']}")
    print(f"  Flags: {row['relevance_flags'][:150]}")


Applying relevance filtering...

Relevance filtering results:
  Total papers: 1877
  Relevant papers (score > 0): 883
  Excluded papers: 994

Saved relevant papers to downloads/processed/
  - relevant_papers.csv
  - relevant_papers.json
  - relevant_papers.xlsx

=== Top 10 Most Relevant Papers ===

14 pts - Survival Analysis of Customer Lifetime and Churn Prediction in the Telecom Industry
  Year: 2025, Source: CrossRef
  Flags: survival:survival analysis; survival:cox proportional hazard; survival:kaplan-meier; survival:survival function; forecasting:customer churn; forecasti

12 pts - Estimating Heterogenous Treatment Effects for Survival Data with Doubly Doubly Robust Estimator
  Year: 2024, Source: arXiv
  Flags: survival:survival analysis; survival:cox proportional hazard; survival:survival function; causal:causal inference; causal:counterfactual; causal:treat

10 pts - LEVERAGING DEEP LEARNING IN SURVIVAL ANALYSIS FOR ENHANCED TIME-TO-EVENT PREDICTION
  Year: 2025, Source: CrossR


Saved relevant papers to downloads/processed/
  - relevant_papers.csv
  - relevant_papers.json
  - relevant_papers.xlsx

=== Top 10 Most Relevant Papers ===

14 pts - Survival Analysis of Customer Lifetime and Churn Prediction in the Telecom Industry
  Year: 2025, Source: CrossRef
  Flags: survival:survival analysis; survival:cox proportional hazard; survival:kaplan-meier; survival:survival function; forecasting:customer churn; forecasti

12 pts - Estimating Heterogenous Treatment Effects for Survival Data with Doubly Doubly Robust Estimator
  Year: 2024, Source: arXiv
  Flags: survival:survival analysis; survival:cox proportional hazard; survival:survival function; causal:causal inference; causal:counterfactual; causal:treat

10 pts - e-Profits: A Business-Aligned Evaluation Metric for Profit-Sensitive Customer Churn Prediction
  Year: 2025, Source: arXiv
  Flags: survival:survival analysis; survival:kaplan-meier; forecasting:customer churn; forecasting:churn prediction; forecasting:

# Create additional analysis files for the systematic review

In [30]:

# Create additional analysis files for the systematic review

# 1. Author analysis
print("Analyzing authors...")
all_authors = []
for idx, row in relevant_df.iterrows():
    authors_str = str(row.get('authors', ''))
    if authors_str and authors_str != 'nan':
        authors = [a.strip() for a in authors_str.split(';') if a.strip()]
        for author in authors:
            all_authors.append({
                'author': author,
                'year': row['year'],
                'paper_title': row['title'],
                'source': row['source'],
                'relevance_score': row['relevance_score']
            })

authors_df = pd.DataFrame(all_authors)
author_counts = authors_df['author'].value_counts().head(50)

author_summary = pd.DataFrame({
    'author': author_counts.index,
    'paper_count': author_counts.values
})
author_summary.to_csv("downloads/processed/top_authors.csv", index=False)
print(f"  Identified {len(authors_df['author'].unique())} unique authors")
print(f"  Top author: {author_counts.index[0]} ({author_counts.values[0]} papers)")

# 2. Journal/Venue analysis
print("\nAnalyzing venues...")
venue_counts = relevant_df[relevant_df['journal'].notna()]['journal'].value_counts().head(30)
venue_df = pd.DataFrame({
    'venue': venue_counts.index,
    'paper_count': venue_counts.values
})
venue_df.to_csv("downloads/processed/top_venues.csv", index=False)
print(f"  Top venue: {venue_counts.index[0]} ({venue_counts.values[0]} papers)")

# 3. Temporal trends
print("\nAnalyzing temporal trends...")
year_trends = relevant_df.groupby(['year', 'source']).size().reset_index(name='count')
year_trends.to_csv("downloads/processed/temporal_trends.csv", index=False)

# Overall year trend
yearly_totals = relevant_df['year'].value_counts().sort_index()
print("  Year-by-year counts:")
for year, count in yearly_totals.items():
    print(f"    {int(year)}: {count} papers")

# 4. Keyword/topic analysis from relevance flags
print("\nAnalyzing topics...")
all_flags = []
for flags_str in relevant_df['relevance_flags']:
    if flags_str and flags_str != 'nan':
        flags = [f.strip() for f in str(flags_str).split(';') if f.strip() and not f.startswith('EXCLUDE')]
        all_flags.extend(flags)

topic_counts = pd.Series(all_flags).value_counts().head(30)
topic_df = pd.DataFrame({
    'topic_flag': topic_counts.index,
    'frequency': topic_counts.values
})
topic_df.to_csv("downloads/processed/topic_frequency.csv", index=False)

print(f"  Most common topic: {topic_counts.index[0]} ({topic_counts.values[0]} occurrences)")

print("\n=== All analysis files created ===")
print("downloads/processed/")
print("  - combined_bibliometric_data.csv/json/xlsx (all papers)")
print("  - relevant_papers.csv/json/xlsx (filtered papers)")
print("  - dataset_summary.json")
print("  - top_authors.csv")
print("  - top_venues.csv")
print("  - temporal_trends.csv")
print("  - topic_frequency.csv")


Analyzing authors...
  Identified 1805 unique authors
  Top author: Durai Rajamanickam (6 papers)

Analyzing venues...
  Top venue:  (256 papers)

Analyzing temporal trends...
  Year-by-year counts:
    2020: 72 papers
    2021: 82 papers
    2022: 99 papers
    2023: 110 papers
    2024: 253 papers
    2025: 243 papers
    2026: 24 papers

Analyzing topics...
  Most common topic: causal:causal inference (181 occurrences)

=== All analysis files created ===
downloads/processed/
  - combined_bibliometric_data.csv/json/xlsx (all papers)
  - relevant_papers.csv/json/xlsx (filtered papers)
  - dataset_summary.json
  - top_authors.csv
  - top_venues.csv
  - temporal_trends.csv
  - topic_frequency.csv
